# v5训练期内收缩控制

只用2018/2019成熟标签，内层2019下半年选择固定网格收缩系数，再训练最终Ridge并一次检验2020。2021/2022不参与；不是新特征或候选账户。

In [1]:
import os,json,hashlib
from pathlib import Path
os.environ["MLFLOW_ALLOW_FILE_STORE"]="true"
os.environ["MLFLOW_DISABLE_AGENT_HINT"]="1"
root=Path.cwd();card=root/"research/stage1_v5_training_only_shrinkage_control_20261030"
if (card/"summary.json").exists():
    import qlib
    from src.core.recorder_sources import load_verified_recorder_objects
    summary=json.loads((card/"summary.json").read_text())
    assert summary["plan_sha256"]==hashlib.sha256((card/"plan.json").read_bytes()).hexdigest()
    qlib.init(provider_uri=str(root/"data/qlib/stage1_event_guard_fullminute_20261028"),region="us",kernels=1)
    saved,receipt=load_verified_recorder_objects(card,expected_recorder_id=summary["recorder_id"],
        artifacts=["models/inner_control","models/final_control","analysis/predictions","analysis/summary"],
        expected_manifest_sha256=hashlib.sha256((card/"verification_manifest.json").read_bytes()).hexdigest())
    assert saved["analysis/summary"]["metrics"]["selected_lambda"]==0
    print("verified_existing_shrinkage_recorder",summary["recorder_id"])
else:
    import os
    os.environ["MLFLOW_ALLOW_FILE_STORE"]="true"
    os.environ["MLFLOW_DISABLE_AGENT_HINT"]="1"
    import hashlib,json
    from pathlib import Path
    import numpy as np
    import pandas as pd
    import qlib
    from qlib.data import D
    from qlib.data.dataset import DatasetH
    from qlib.data.dataset.handler import DataHandlerLP
    from qlib.workflow import R
    from sklearn.impute import SimpleImputer
    from sklearn.preprocessing import StandardScaler
    from sklearn.linear_model import Ridge
    from sklearn.pipeline import make_pipeline
    from src.core.recorder_sources import load_verified_recorder_objects
    from src.workflow.minute_action_value import _source
    root=Path.cwd();card=root/"research/stage1_v5_training_only_shrinkage_control_20261030"
    plan_bytes=(card/"plan.json").read_bytes();plan=json.loads(plan_bytes);sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
    label_card=root/plan["label_source"]["card"]
    assert sha(label_card/"summary.json")==plan["label_source"]["summary_sha256"]
    control_card=root/plan["old_control_source"]["card"]
    assert sha(control_card/"summary.json")==plan["old_control_source"]["summary_sha256"]
    assert sha(control_card/"verification_manifest.json")==plan["old_control_source"]["manifest_sha256"]
    assert json.loads((control_card/"nas_binding.json").read_text())["evidence"]["sha256"]==plan["old_control_source"]["archive_sha256"]
    assert sha(label_card/"verification_manifest.json")==plan["label_source"]["manifest_sha256"]
    assert json.loads((label_card/"nas_binding.json").read_text())["evidence"]["sha256"]==plan["label_source"]["nas_archive_sha256"]
    assert sha(root/plan["provider_uri"]/"qlib_store_manifest.json")==plan["provider_manifest_sha256"]
    qlib.init(provider_uri=str(root/plan["provider_uri"]),region="us",kernels=1)
    label_objects,label_receipt=load_verified_recorder_objects(label_card,expected_recorder_id=plan["label_source"]["recorder_id"],
        artifacts=["analysis/labels","analysis/audit"],expected_manifest_sha256=plan["label_source"]["manifest_sha256"])
    labels=label_objects["analysis/labels"]
    assert len(labels)==label_objects["analysis/audit"]["event_slots"]
    labels=labels.loc[labels.trade_date.dt.year.le(2020) & labels.soxl_funded & labels.soxx_funded].copy()
    source,source_receipt=_source(root,plan["training_source"],["analysis/native/dated_f696_0"])
    frame_source,frame_receipt=_source(root,plan["frame_source"],["inputs/covered_frame"])
    book=source["analysis/native/dated_f696_0"]
    frame=frame_source["inputs/covered_frame"].reset_index(drop=True)
    assert frame.trade_date.equals(book["authority"].trade_date)
    frame=frame.loc[frame.trade_date.dt.year.le(2020)].copy()
    assert list(frame.columns[3:42])==plan["base39_features"]
    assert frame.feature_available_at.lt(frame.trade_date+pd.Timedelta(hours=9,minutes=46)).all()
    state=frame[["trade_date",*plan["base39_features"],"soxl_weight","soxx_weight","cash_weight","tail_probability","alpha_probability"]].copy()
    state["risk_state"]=book["authority"].loc[book["authority"].trade_date.dt.year.le(2020),"risk_state"].to_numpy()
    state["risk_clock"]=book["authority"].loc[book["authority"].trade_date.dt.year.le(2020),"risk_clock"].to_numpy()
    quant=[]
    for day in state.trade_date:
     stamp=pd.Timestamp(day)+pd.Timedelta(hours=9,minutes=47)
     pos=book["native_positions"].get(stamp)
     assert pos is not None
     quant.append([pos.get_stock_amount(s) if pos.check_stock(s) else 0. for s in ("SOXL","SOXX")])
    state[["q_soxl","q_soxx"]]=np.asarray(quant,float)
    paired=labels.merge(state,on="trade_date",how="left",validate="many_to_one")
    assert paired[plan["base39_features"]].notna().all().all()
    bars=D.features(["SOXL","SOXX"],["$close","$raw_volume","$price_valid"],start_time="2018-10-15 09:31:00",end_time="2020-12-31 16:00:00",freq="1min")
    parts=[]
    for symbol in ("SOXL","SOXX"):
     part=bars.xs(symbol,level="instrument").copy()
     part=part.loc[part.index.normalize().isin(pd.DatetimeIndex(paired.trade_date.unique()))]
     assert part["$price_valid"].eq(1).all()
     days=part.index.normalize();logp=np.log(part["$close"].astype(float))
     out=pd.DataFrame(index=part.index)
     for width in (5,15):out[f"minute_{symbol.lower()}_ret{width}"]=logp-logp.groupby(days).shift(width)
     out[f"minute_{symbol.lower()}_log1p_raw_volume"]=np.log1p(part["$raw_volume"].astype(float))
     out[f"{symbol.lower()}_mark"]=part["$close"].astype(float)
     parts.append(out)
    minute=parts[0].join(parts[1],how="inner")
    paired=paired.join(minute,on="bar_end_at",how="left")
    assert paired[["soxl_mark","soxx_mark"]].notna().all().all()
    paired["decision_soxl_weight"]=paired.q_soxl*paired.soxl_mark/paired.decision_nav
    paired["decision_soxx_weight"]=paired.q_soxx*paired.soxx_mark/paired.decision_nav
    paired["clock_fraction"]=(paired.bar_end_at.dt.hour*60+paired.bar_end_at.dt.minute-(9*60+30))/390.
    assert paired.label_available_at.gt(paired.decision_at).all()
    assert paired.trade_date.dt.year.isin([2018,2019,2020]).all()
    state_features=plan["base39_features"]+plan["state_features"]
    full_features=state_features+plan["ordinary_minute_features"]
    assert len(set(full_features))==len(full_features)
    packed=pd.concat({"feature":paired[full_features],"label":paired[[plan["primary_label"]]],
        "metadata":paired[["trade_date","decision_at","label_available_at"]]},axis=1)
    packed.index=pd.MultiIndex.from_arrays([paired.decision_at,["SOXL_SOXX_CASH"]*len(paired)],names=["datetime","instrument"])
    dataset=DatasetH(handler=DataHandlerLP.from_df(packed),segments={"train":("2018-10-15","2019-12-31"),"test":("2020-01-01","2020-12-31")})
    train=dataset.prepare("train",col_set=["feature","label","metadata"])
    test=dataset.prepare("test",col_set=["feature","label","metadata"])
    first_test=pd.Timestamp(test.index.get_level_values("datetime").min())
    train=train.loc[pd.to_datetime(train[("metadata","label_available_at")]).lt(first_test)].copy()
    assert train.index.get_level_values("datetime").max()<first_test
    assert train[("metadata","trade_date")].dt.year.isin([2018,2019]).all()
    assert test[("metadata","trade_date")].dt.year.eq(2020).all()
    ytrain=train[("label",plan["primary_label"])].to_numpy(float)
    ytest=test[("label",plan["primary_label"])].to_numpy(float)
    train_days=pd.to_datetime(train[("metadata","trade_date")]).to_numpy()
    test_days=pd.to_datetime(test[("metadata","trade_date")]).to_numpy()
    train_n=pd.Series(train_days).map(pd.Series(train_days).value_counts()).to_numpy(float)
    weights=1./train_n
    
    
    old_card=root/plan["old_control_source"]["card"]
    assert sha(old_card/"summary.json")==plan["old_control_source"]["summary_sha256"]
    assert sha(old_card/"verification_manifest.json")==plan["old_control_source"]["manifest_sha256"]
    old_objects,old_receipt=load_verified_recorder_objects(old_card,expected_recorder_id=plan["old_control_source"]["recorder_id"],
        artifacts=["models/ordinary_minute","analysis/predictions"],expected_manifest_sha256=plan["old_control_source"]["manifest_sha256"])
    first_val=train.loc[pd.to_datetime(train[("metadata","trade_date")]).ge(pd.Timestamp("2019-07-01"))].index.get_level_values("datetime").min()
    inner_train=train.loc[(train.index.get_level_values("datetime")<first_val)&
        pd.to_datetime(train[("metadata","label_available_at")]).lt(first_val)].copy()
    inner_val=train.loc[train.index.get_level_values("datetime")>=first_val].copy()
    assert inner_train[("metadata","trade_date")].dt.year.isin([2018,2019]).all()
    assert inner_val[("metadata","trade_date")].dt.year.eq(2019).all()
    assert inner_train.index.get_level_values("datetime").max()<first_val
    features=full_features
    
    def daily_weights(data):
        days=pd.to_datetime(data[("metadata","trade_date")]).to_numpy()
        count=pd.Series(days).map(pd.Series(days).value_counts()).to_numpy(float)
        return 1/count
    
    def fit_control(data):
        x=data.loc[:,pd.IndexSlice["feature",features]]
        y=data[("label",plan["primary_label"])].to_numpy(float)
        model=make_pipeline(SimpleImputer(strategy="median",add_indicator=True),StandardScaler(),Ridge(alpha=1.0))
        model.fit(x,y,ridge__sample_weight=daily_weights(data))
        return model
    
    def day_mse(data,values):
        days=pd.to_datetime(data[("metadata","trade_date")]).to_numpy()
        y=data[("label",plan["primary_label"])].to_numpy(float)
        return float(pd.DataFrame({"day":days,"error":(y-values)**2}).groupby("day").error.mean().mean())
    
    inner_model=fit_control(inner_train)
    inner_mean=float(np.average(inner_train[("label",plan["primary_label"])].to_numpy(float),weights=daily_weights(inner_train)))
    inner_raw=inner_model.predict(inner_val.loc[:,pd.IndexSlice["feature",features]])
    inner_scores={str(v):day_mse(inner_val,inner_mean+v*(inner_raw-inner_mean)) for v in plan["shrinkage_grid"]}
    selected=min(plan["shrinkage_grid"],key=lambda v:(inner_scores[str(v)],v))
    final_model=fit_control(train)
    final_mean=float(np.average(ytrain,weights=weights))
    raw=final_model.predict(test.loc[:,pd.IndexSlice["feature",features]])
    old_pred=old_objects["analysis/predictions"].ordinary_prediction.to_numpy(float)
    assert np.max(np.abs(raw-old_pred))<=1e-12
    shrunk=final_mean+selected*(raw-final_mean)
    constant=np.full(len(test),final_mean)
    metrics={"selected_lambda":float(selected),"inner_train_days":int(inner_train[("metadata","trade_date")].nunique()),
        "inner_validation_days":int(inner_val[("metadata","trade_date")].nunique()),"outer_train_days":int(pd.Series(train_days).nunique()),
        "outer_eval_days":int(pd.Series(test_days).nunique()),"inner_scores":inner_scores,
        "outer_constant_day_mse":day_mse(test,constant),"outer_raw_ridge_day_mse":day_mse(test,raw),
        "outer_shrunk_day_mse":day_mse(test,shrunk),"raw_saved_prediction_max_error":float(np.max(np.abs(raw-old_pred)))}
    qualified=bool(metrics["outer_shrunk_day_mse"]<metrics["outer_constant_day_mse"]-1e-12)
    metrics["control_residual_qualified"]=qualified
    prediction=pd.DataFrame({"trade_date":test_days,"decision_at":test.index.get_level_values("datetime"),
        "actual_value":ytest,"constant_prediction":constant,"raw_ridge_prediction":raw,
        "shrunk_prediction":shrunk,"shrunk_residual":ytest-shrunk})
    with R.start(experiment_name=card.name,uri=str(card/"qlib_recorder")):
        recorder=R.get_recorder()
        recorder.save_objects(artifact_path="inputs",plan=plan,source_readbacks={"labels":label_receipt,"parent":source_receipt,"frame":frame_receipt,"old_control":old_receipt})
        recorder.save_objects(artifact_path="models",inner_control=inner_model,final_control=final_model)
        recorder.save_objects(artifact_path="analysis",predictions=prediction,metrics=metrics)
        recorder.log_metrics(selected_lambda=float(selected),outer_shrunk_day_mse=metrics["outer_shrunk_day_mse"],outer_constant_day_mse=metrics["outer_constant_day_mse"],control_residual_qualified=float(qualified))
    summary={"experiment":plan["experiment"],"plan_sha256":sha(card/"plan.json"),
        "decision":"training_only_shrinkage_control_pass_needs_residual_analysis" if qualified else "not_admitted_training_only_shrinkage_control_failed_constant",
        "metrics":metrics,"new_model_fits":2,"new_native_accounts":0,"historical_research_passed":False,
        "recorder_id":recorder.id,"experiment_id":str(recorder.experiment_id),"recorder_uri":str(card/"qlib_recorder"),
        "recorder_local_dir":str(card/"qlib_recorder"/str(recorder.experiment_id)/recorder.id),
        "artifact_policy":{"policy":"slim_card_recorder_first","workflow_config":"workflow.yaml","recorder_only_artifacts":["inputs/*","models/*","analysis/*"]},
        "limitations":["Lambda was selected on 2019 inner validation, not2020; 2020 is reused development feedback.","2021/2022 labels were not evaluated.","Funded-pair conditioning is not a live-intent policy."]}
    (card/"summary.json").write_text(json.dumps(summary,ensure_ascii=False,indent=2)+"\n")
    print(json.dumps({"decision":summary["decision"],"metrics":metrics},ensure_ascii=False))

## 中文结论

内层176日训练、2019下半年127日验证，固定λ网格0/0.25/0.5/1按每日等权MSE选择λ=0；其他λ全部较差。外层304日训练、2020年242日评价，λ=0收缩与训练日均常数MSE同为1.240507e-8，原Ridge为1.798565e-8，预先冻结的“严格胜常数”控制门失败。旧Ridge保存预测逐行差0；2021/2022标签没有参与。前三次启动均为拟合前来源/时间实现错误，保留收据；正式一次恰好2次拟合。Qlib Recorder模型、预测与NAS限定远端读回通过。没有新候选特征或完整账户，SOXX目标未过。下一机制转到SOXX相对HOLD现金部署价值，但须与旧固定现金路线实质区分并另冻标签/控制。